## Generador del dataset sintético - La Redoma Rosa (AoT)
---

<img src="../imagenes/opcion_A.png" width="620"/>

---

Ejecuta este script para generar el CSV de la práctica del A/B testing

In [14]:
# Instalamos las librerías pandas y numpy
! pip install pandas numpy


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Importamos librerías
import pandas as pd
import numpy as np

### Semilla de aleatoriedad 🌱
Fijar la semilla garantiza que todos trabajamos con el mismo dataset.   
El número **42** se puede modificar, no tiene nada de especial matemáticamente. Es un guiño a **La Guía del Autoestopista Galáctico (The Hitchhiker's Guide to the Galaxy)**, donde el 42 es "la respuesta a la vida, el universo y todo lo demás". Se ha convertido en el número por defecto en casi todos los ejemplos de data science del mundo.

In [16]:
np.random.seed(42)

### Parámetros del experimento
Podemos modificar estos valores para experimentar con distintos escenarios.    
Vamos a utilizar la convención  SCREAMING_SNAKE_CASE, que en Python se utiliza para indicar que una variable es constante, es decir, un valor que se define una vez al principio y no debería cambiar durante la ejecución del programa.    
Variable en MAYÚSCULA → es un parámetro fijo, ¡no la toques!

In [17]:
N_USUARIOS_POR_VERSION = 4000   # Usuarios asignados a cada versión

TASA_CONVERSION_A = 0.068       # Versión A (Vintage Rosa): 6.8% compran
TASA_CONVERSION_B = 0.044       # Versión B (Moderna):      4.4% compran

### Determinamos los concejos de Asturias en los que la empresa tiene presencia
Los pesos reflejan aproximadamente la población relativa de cada concejo, son proporciones (no porcentajes).

In [18]:
concejos = [
    "Gijón", "Oviedo", "Avilés", "Siero", "Langreo", "Mieres",
    "Castrillón", "Teverga", "Gozón", "Carreño", "Villaviciosa",
    "San Martín del Rey Aurelio", "Salas", "Cangas del Narcea", "Piloña", "Nava",
    "Cudillero", "Luarca", "Lena", "Noreña"
]

pesos = [
    0.26, 0.22, 0.10, 0.07, 0.06, 0.05,
    0.04, 0.01, 0.02, 0.02, 0.02,
    0.02, 0.01, 0.02, 0.01, 0.01,
    0.01, 0.01, 0.01, 0.01
]

Normalizamos los pesos, la suma de la porporción de los pesos ha de ser *1*.

In [19]:
pesos = np.array(pesos) / sum(pesos)

Comprobamos que hayamos definido bien los concejos y los pesos.

In [20]:
print(f"Número de concejos : {len(concejos)}")
print(f"Número de pesos    : {len(pesos)}")
print(f"Suma de pesos      : {sum(pesos):.4f}")  # tiene que ser 1.0

Número de concejos : 20
Número de pesos    : 20
Suma de pesos      : 1.0000


### Establecemos los productos del catálogo

In [21]:
productos = [
    "Aceite de argán artesanal",
    "Champú sólido de romero",
    "Crema de lavanda y miel",
    "Mascarilla de arcilla verde",
    "Sérum de rosa mosqueta",
    "Jabón de avena y caléndula",
    "Acondicionador de ortiga",
    "Bálsamo labial de cera de abeja"
]

Comprobamos que se haya ejecutado correctamente la definición de los productos.

In [22]:
print(f"Número de productos: {len(productos)}")
print(*productos, sep='\n')  # los lista uno por línea para revisar tildes y erratas

Número de productos: 8
Aceite de argán artesanal
Champú sólido de romero
Crema de lavanda y miel
Mascarilla de arcilla verde
Sérum de rosa mosqueta
Jabón de avena y caléndula
Acondicionador de ortiga
Bálsamo labial de cera de abeja


### Función para generar un grupo

In [23]:
def generar_grupo(version, tasa, n):
    """
    Genera los registros de un grupo del experimento.

    Parámetros:
        version  : 'A' o 'B'
        tasa     : tasa de conversión real de ese grupo
        n        : número de usuarios a generar

    Devuelve:
        DataFrame con una fila por usuario
    """
    # Los IDs de A van de 1 a N, los de B de N+1 a 2N
    # (así no se repiten entre grupos)
    inicio_id = 1 if version == 'A' else n + 1
    ids = np.arange(inicio_id, inicio_id + n)

    return pd.DataFrame({
        'usuario_id'        : ids,
        'version'           : version,
        'pagina'            : 'vintage_rosa' if version == 'A' else 'moderna',
        'concejo'         : np.random.choice(concejos, size=n, p=pesos), # elige aleatoriamnte n concejos de nuestra lista
        'producto_visto'    : np.random.choice(productos, size=n), # elige n productos, todos con la misma probabilidad
        'tiempo_sesion_seg' : np.random.randint(30, 420, size=n), # genera n números enteros aleatorios entre 30 y 420 — los segundos que cada usuario pasó en la web.
        # np.random.binomial(1, p) devuelve 1 (compra) o 0 (no compra)
        # con probabilidad p — simula la decisión de cada usuario
        'convirtio'         : np.random.binomial(1, tasa, size=n) # 'convirtio' es jerga de marketing digital, equivale a "el usuario realizó la acción objetivo".
    })


### Generamos los datos

In [24]:
df_a = generar_grupo('A', TASA_CONVERSION_A, N_USUARIOS_POR_VERSION)
df_b = generar_grupo('B', TASA_CONVERSION_B, N_USUARIOS_POR_VERSION)

# Mezclamos las filas de ambos grupos aleatoriamente
# (como llegarían en la realidad, intercalados en el tiempo)
df = pd.concat([df_a, df_b], ignore_index=True) #'concat' concatena los dataframes e 'ignore_index=True' hace que los números, los índices, no se repitan.
df = df.sample(frac=1, random_state=42).reset_index(drop=True) # 'sample' coge una muestra aleatoria del DataFrame. 'frac=1' significa "coge el 100% de las filas", es decir, todas, pero en orden aleatorio

In [25]:
print(df['version'].value_counts())           # ¿los dos grupos tienen el mismo tamaño?
print(df['concejo'].value_counts(normalize=True).head())  # ¿los concejos salen en la proporción esperada?
print(df['convirtio'].mean())                 # ¿la tasa de conversión global tiene sentido?
print(df.isnull().sum())                      # ¿hay valores vacíos?

version
A    4000
B    4000
Name: count, dtype: int64
concejo
Gijón      0.265000
Oviedo     0.224750
Avilés     0.103500
Siero      0.074125
Langreo    0.062250
Name: proportion, dtype: float64
0.05525
usuario_id           0
version              0
pagina               0
concejo              0
producto_visto       0
tiempo_sesion_seg    0
convirtio            0
dtype: int64


### Guardamos el csv

In [26]:
nombre_fichero = 'la_redoma_rosa.csv'
df.to_csv(nombre_fichero, index=False)

## Resumen

In [27]:
print(f"Dataset generado: {len(df):,} registros")
print(f"Fichero guardado: {nombre_fichero}")
print()
print(f"Tasa de conversión A (Vintage Rosa): {df_a['convirtio'].mean():.2%}")
print(f"Tasa de conversión B (Moderna)     : {df_b['convirtio'].mean():.2%}")
print()
print("Distribución por concejo (top 5):")
print(df['concejo'].value_counts().head())
print()
print("Primeras filas del dataset:")
print(df.head(5).to_string())

Dataset generado: 8,000 registros
Fichero guardado: la_redoma_rosa.csv

Tasa de conversión A (Vintage Rosa): 6.60%
Tasa de conversión B (Moderna)     : 4.45%

Distribución por concejo (top 5):
concejo
Gijón      2120
Oviedo     1798
Avilés      828
Siero       593
Langreo     498
Name: count, dtype: int64

Primeras filas del dataset:
   usuario_id version        pagina            concejo                   producto_visto  tiempo_sesion_seg  convirtio
0        2216       A  vintage_rosa             Oviedo         Acondicionador de ortiga                158          0
1        2583       A  vintage_rosa             Avilés           Sérum de rosa mosqueta                286          0
2        1663       A  vintage_rosa  Cangas del Narcea  Bálsamo labial de cera de abeja                375          0
3        3028       A  vintage_rosa              Siero         Acondicionador de ortiga                144          0
4        4344       B       moderna            Langreo          Champú sól